In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Get started with the workshop and the agent platform

| | |
|-|-|
| Author(s) | [Matt Robinson](https://github.com/mr394729) |

> **This copy keeps the output of one complete run** (24 September 2026, in a test namespace), so you can read what each cell prints even if a cell fails for you. Your numbers and wording will differ where a model answers. To start clean, choose **Edit > Clear Outputs of All Cells** in JupyterLab, or run `jupyter nbconvert --clear-output --inplace <notebook>`.

## Overview

### The workshop

Over three hours you build, evaluate, deploy and govern one agent: an assistant for the team in a Cymbal Beauty store. Each notebook covers one part of the session.

| Time | Session | Notebook |
|---|---|---|
| 0:00–0:45 | Welcome; build, scale, govern and optimize | 00 |
| 0:45–1:00 | Store operations and the data | 01 |
| 1:00–1:30 | ADK and agent patterns | 02 |
| 1:30–2:00 | Tools and workflows | 03 |
| 2:00–2:15 | Evaluation and observability | 04 |
| 2:15–2:30 | Deployment and promotion | 05 |
| 2:30–2:50 | Governance | 06 |
| 2:50–3:00 | Cost and tokens | 07 |

### The agent platform

The workshop uses four layers of Google Cloud's agent platform:

- **Build** with [Agent Development Kit](https://google.github.io/adk-docs/) (ADK), an open-source framework for agents, their tools and their workflows.
- **Scale** with [Agent Runtime](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/overview) (formerly Vertex AI Agent Engine), a managed service that hosts the agent, its sessions and its memory.
- **Govern** with identity, access controls, [Model Armor](https://cloud.google.com/security-command-center/docs/model-armor-overview) content screening and a registry of agents and tools.
- **Optimize** with evaluation, tracing and token accounting.

The store's data lives in [BigQuery](https://cloud.google.com/bigquery/docs/introduction).

<img width="60%" src="../docs/diagrams/platform.png" alt="The agent platform: build, scale, govern and optimize" />

### The store agent

The agent helps two people. Dana, the store manager, gets an opening plan, staffing and pickup coverage, shelf availability, loss reviews and coaching. Priya, an associate, gets her own tasks, stock locations and product advice. Every change to a store task asks for approval first.

<img width="60%" src="../docs/diagrams/capabilities.png" alt="What the store agent does for a manager and an associate" />

### Objectives

In this tutorial, you will set up your workspace and check that it can reach each part of the platform.

You will complete the following tasks:

- Check the Python packages in your kernel
- Call Gemini on Vertex AI
- Read the agent's configuration for each environment
- Send a question to your deployed agent on Agent Runtime

### Costs

This tutorial uses billable components of Google Cloud:

- Gemini on Vertex AI
- Agent Runtime on Vertex AI

Learn about [Vertex AI pricing](https://cloud.google.com/vertex-ai/pricing) and use the [Pricing Calculator](https://cloud.google.com/products/calculator/) to generate a cost estimate based on your projected usage.

## Get started

### Before you begin

Complete the [cloud setup](../SETUP.md) in a terminal first: install, sign in and choose your namespace. You need:

- **A Google Cloud project.** In the workshop everyone shares one project that the facilitator has prepared: the APIs are enabled (Vertex AI, BigQuery, Cloud Run, Secret Manager, Model Armor and the others in the [shared project guide](../docs/SHARED_PROJECT.md)) and your account has the roles listed there. With your own project, follow that guide first.
- **A namespace.** Three to twelve lowercase letters and digits, starting with a letter, such as your initials. Every resource you create carries it.
- **On your machine:** Python 3.12, [uv](https://docs.astral.sh/uv/), Git and the [Google Cloud CLI](https://cloud.google.com/sdk/docs/install).

### Install packages

The workshop repository pins every package in `uv.lock`. From the repository root, in a terminal, install them and start Jupyter:

```bash
uv sync --all-extras
uv run jupyter lab notebooks/
```

Check that this notebook runs with the repository's environment. The labs are written for ADK 2.9:

In [2]:
from importlib.metadata import version

for package in ["google-adk", "google-genai", "google-cloud-aiplatform", "google-cloud-bigquery"]:
    print(f"{package:26} {version(package)}")

google-adk                 2.9.0
google-genai               2.23.0
google-cloud-aiplatform    1.165.1
google-cloud-bigquery      3.45.0


### Authenticate your notebook environment

In a terminal, sign in twice: once for the `gcloud` command-line tool, and once for the Python client libraries this notebook uses (Application Default Credentials).

```bash
gcloud auth login
gcloud auth application-default login
```

### Set Google Cloud project information

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

Everyone in the workshop shares one project, so each person picks a short namespace, such as their initials. Your BigQuery dataset, your deployed agent and your templates all carry it, so nobody overwrites anybody else's work.

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [3]:
import os
import sys
from pathlib import Path

PROJECT_ID = "[your-project-id]"  # @param {type: "string"}
WORKSHOP_NAMESPACE = "[your-namespace]"  # @param {type: "string"}

if PROJECT_ID == "[your-project-id]":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "")
if WORKSHOP_NAMESPACE == "[your-namespace]":
    WORKSHOP_NAMESPACE = os.environ.get("WORKSHOP_NAMESPACE", "")
if not PROJECT_ID or not WORKSHOP_NAMESPACE:
    raise ValueError("Set PROJECT_ID and WORKSHOP_NAMESPACE above.")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["WORKSHOP_NAMESPACE"] = WORKSHOP_NAMESPACE
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["STORE_OPS_ENV"] = "dev"

# The agent's code lives one folder up from this notebook
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

### Import libraries

In [4]:
import json
import logging
import warnings

import pandas as pd
import requests
import vertexai
import yaml
from google import genai
from google.auth import default as google_auth_default
from google.auth.transport.requests import Request

# Keep the output to the results: the Vertex AI SDK announces its newer package name with a
# FutureWarning, and the Gen AI SDK logs a note about automatic function calling on every request.
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("google_genai").setLevel(logging.ERROR)

## Call Gemini

Send one prompt to Gemini through Vertex AI. The agent uses `gemini-3.8-flash` from the `global` endpoint, so this call checks your credentials, your project and your access to the model in one step.

In [5]:
client = genai.Client(vertexai=True, project=PROJECT_ID, location="global")

response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="In one sentence, what does an assistant for a beauty store team do?",
)
print(response.text)

A beauty store assistant supports the team by providing personalized customer service and product recommendations, processing transactions, and maintaining organized, well-stocked merchandise displays.


The response also reports how many tokens the request used. Notebook 07 turns these counts into cost:

In [6]:
usage = response.usage_metadata
print(f"Prompt tokens:    {usage.prompt_token_count}")
print(f"Output tokens:    {usage.candidates_token_count}")
print(f"Thinking tokens:  {usage.thoughts_token_count}")

Prompt tokens:    15
Output tokens:    29
Thinking tokens:  399


Thinking tokens are the tokens the model spent reasoning before it answered. In this run it used 399 of them for a 29-token answer. The counts change from run to run.

## Read the agent's configuration

Each environment (dev, preprod and prod) has one small YAML file. The model and the thinking levels are the same in every environment, so the agent you evaluate in dev is the agent that runs in prod. Only the runtime settings, such as the number of warm instances, change.

The agent thinks at a medium level, except for the step that writes the opening plan, which runs at a low level: in testing that made the opening plan about four times faster with no loss of accuracy.

In [7]:
config_dir = REPO_ROOT / "agents" / "cymbal_store_ops" / "config" / "envs"

rows = []
for path in sorted(config_dir.glob("*.yaml")):
    config = yaml.safe_load(path.read_text())
    rows.append(
        {
            "environment": path.stem,
            "model": config["model"],
            "model_location": config["model_location"],
            "thinking_level": config["thinking_level"],
            "plan_writer_thinking_level": config.get("plan_writer_thinking_level"),
            "min_instances": config["agent_engine"]["min_instances"],
        }
    )

pd.DataFrame(rows)

,environment,model,model_location,thinking_level,plan_writer_thinking_level,min_instances
0,dev,gemini-3.8-flash,global,medium,low,1
1,preprod,gemini-3.8-flash,global,medium,low,0
2,prod,gemini-3.8-flash,global,medium,low,1


All three environments use `gemini-3.8-flash` on the `global` endpoint, with medium thinking and a low level for the plan writer. Only `min_instances` differs: dev and prod keep one warm instance, and preprod scales to zero.

## Query your deployed agent

If you deployed the agent before the workshop, it is running on Agent Runtime as `cymbal-store-ops-<namespace>-dev`. If you have not deployed yet, skip this section: notebook 05 deploys it.

### Find your agent

Agent Runtime is a regional service. Create a Vertex AI client for `us-central1` and look up your agent by its display name:

In [8]:
vertex_client = vertexai.Client(project=PROJECT_ID, location="us-central1")

display_name = f"cymbal-store-ops-{WORKSHOP_NAMESPACE}-dev"
matches = list(
    vertex_client.agent_engines.list(config={"filter": f'display_name="{display_name}"'})
)
if not matches:
    raise ValueError(f"No agent named {display_name}. Deploy it in notebook 05 first.")

remote_agent = vertex_client.agent_engines.get(name=matches[0].api_resource.name)
print(remote_agent.api_resource.name)

projects/763419985448/locations/us-central1/reasoningEngines/7051567741404184576


### Create a session

A session holds one conversation. The agent reads who is signed in from the session state, so create the session as Dana, the manager of store S-014:

In [9]:
session = remote_agent.create_session(
    user_id="dana",
    state={
        "user:user_id": "U-M014",
        "user:store_id": "S-014",
        "user:role": "store_manager",
        "user:first_name": "Dana",
    },
)
session_id = session["id"] if isinstance(session, dict) else session.id
print(session_id)

6377945105059282944


### Send a question

Send a question to the agent's [`streamQuery` REST method](https://cloud.google.com/vertex-ai/generative-ai/docs/reference/rest/v1/projects.locations.reasoningEngines/streamQuery). The response is a stream of ADK events, one JSON object per line: tool calls, tool results and the final answer. This cell prints the tool calls and the answer. It takes about 20 seconds.

In [10]:
credentials, _ = google_auth_default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
credentials.refresh(Request())

response = requests.post(
    f"https://us-central1-aiplatform.googleapis.com/v1/{remote_agent.api_resource.name}:streamQuery",
    headers={"Authorization": f"Bearer {credentials.token}"},
    json={
        "class_method": "async_stream_query",
        "input": {
            "user_id": "dana",
            "session_id": session_id,
            "message": "How many Lumière Hydra Cream units do we have on hand?",
        },
    },
    stream=True,
    timeout=300,
)
response.raise_for_status()

for line in response.iter_lines(decode_unicode=True):
    if not line:
        continue
    event = json.loads(line)
    for part in (event.get("content") or {}).get("parts", []):
        if part.get("function_call"):
            print(f"[{event['author']}] calls {part['function_call']['name']}")
        if part.get("text") and not part.get("thought"):
            print(f"\n{part['text']}")

[store_manager_agent] calls list_store_inventory



We have 7 units of Lumière Hydra Cream (SKU: P-0101) on hand. All 7 units are currently recorded in the backroom, with 0 units on the shelf. Please note that this reflects gross recorded stock balances rather than reservation-adjusted availability.


The agent called one tool, `list_store_inventory`, and found 7 units of Lumière Hydra Cream (P-0101), all in the backroom and none on the shelf. The wording differs from run to run; the tool call and the numbers should match.

## Cleaning up

This notebook created one session on your agent. Delete it:

In [11]:
remote_agent.delete_session(user_id="dana", session_id=session_id)

## What's next

- [Agent Runtime overview](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/overview)
- [Use an ADK agent on Agent Runtime](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/use/adk)
- [Cloud setup for the workshop](../SETUP.md)
- Next notebook: [Explore and load the store data](01_store_data.ipynb)